# Weak evidence at very small $p$: the Schmidt–Jahn–Radin psychokinesis experiment

Schmidt, Jahn and Radin (1987) ran a quantum-mechanical random event generator: each "particle" passes a quantum gate and lights up red ($1$) or green ($0$). With no influence, each trial is Bernoulli with $\theta = 1/2$. A subject who claims psychokinetic ability tries to bias the generator toward red.

Across $n = 104{,}900{,}000$ trials, the observed number of red lights was
$$X_{\text{obs}} \;=\; 52{,}468{,}000,$$
a deviation of about $+18{,}000$ above the $\theta = 0.5$ baseline of $n/2 = 52{,}450{,}000$ — a deviation of about three-and-a-half standard deviations and, on classical grounds, "highly significant."

We compare two analyses of these data:

- **Frequentist.** Test $H_0\colon \theta = 1/2$ against $H_1\colon \theta > 1/2$. The one-sided $p$-value comes out around $0.0003$, well below any conventional threshold.
- **Bayesian (Jefferys 1990, default prior).** Place equal prior weight on $H_0\colon \theta = 1/2$ and $H_1\colon \theta \sim U(0,1)$. Compute $\Pr(H_0 \mid X \approx X_{\text{obs}})$ by Monte Carlo.

The paradox: a $p$-value below $10^{-3}$ coexists with a posterior probability $\Pr(H_0 \mid x) \approx 0.94$. A "highly significant" $p$-value can still leave $H_0$ as overwhelmingly the more probable hypothesis.

In [1]:
import numpy as np
from scipy.stats import norm

rng = np.random.default_rng(42)

n = 104_900_000           # number of Bernoulli trials in the experiment
X_obs = 52_468_000        # observed number of red lights

## The classical $p$-value under $H_0$

Under $H_0\colon \theta = 1/2$, $X \sim \text{Binomial}(n, 0.5)$ with mean $n/2$ and standard deviation $\sqrt{n/4}$. With $n$ this large, the Normal approximation is excellent. The one-sided $p$-value is
$$p \;=\; P(X \ge X_{\text{obs}} \mid H_0) \;=\; 1 - \Phi\!\left(\frac{X_{\text{obs}} - n/2}{\sqrt{n/4}}\right).$$

In [2]:
mu_H0 = n / 2
sigma_H0 = np.sqrt(n / 4)
z_obs = (X_obs - mu_H0) / sigma_H0
p_value = 1 - norm.cdf(z_obs)

print(f"n                  = {n:,}")
print(f"X_obs              = {X_obs:,}")
print(f"mean under H_0     = {mu_H0:,.0f}")
print(f"SD under H_0       = {sigma_H0:,.2f}")
print(f"z-stat at X_obs    = {z_obs:.3f}")
print(f"one-sided p-value  = {p_value:.6f}")

n                  = 104,900,000
X_obs              = 52,468,000
mean under H_0     = 52,450,000
SD under H_0       = 5,121.04
z-stat at X_obs    = 3.515
one-sided p-value  = 0.000220


## Monte Carlo: posterior $\Pr(H_0 \mid X \approx X_{\text{obs}})$

Draw $\theta$ from the prior mixture (50/50 over $H_0$ and $H_1$), then $X \mid \theta \sim N(n\theta,\; n\theta(1-\theta))$ (Normal approximation to the binomial). Filter on a tight window around $X_{\text{obs}}$ — wide enough that we get a sample size in the hundreds, narrow enough that the likelihood is essentially constant across the window. The empirical share of those trials that came from $H_0$ is the Monte Carlo estimate of $\Pr(H_0 \mid X \approx X_{\text{obs}})$.

In [3]:
M = 5_000_000

# Draw theta from the prior mixture: H_0 (theta = 1/2) with weight 1/2, H_1 (theta ~ U[0,1]) with weight 1/2.
H1 = rng.random(M) >= 0.5
theta = np.where(H1, rng.random(M), 0.5)

# Draw X | theta ~ N(n*theta, n*theta*(1-theta)) (Normal approx to Binomial).
mean_X = n * theta
sd_X = np.sqrt(n * theta * (1 - theta))
X = rng.normal(mean_X, sd_X)

In [4]:
# Filter: |X - X_obs| <= 0.1 * sigma_H0. Tight enough that the likelihood is
# nearly constant across the window, wide enough to give us hundreds of hits.
window = 0.1 * sigma_H0
near_obs = np.abs(X - X_obs) <= window

n_H0 = (~H1 & near_obs).sum()
n_H1 = ( H1 & near_obs).sum()
n_total = near_obs.sum()

post_p_H0 = n_H0 / n_total
post_odds_H0_to_H1 = n_H0 / n_H1

print(f"MC trials                          = {M:,}")
print(f"filter window (X units)            = {window:,.2f}")
print(f"trials with X near X_obs           = {n_total:,}")
print(f"  ... from H_0 (theta = 1/2)       = {n_H0:,}")
print(f"  ... from H_1 (theta ~ U[0,1])    = {n_H1:,}")
print(f"posterior P(H_0 | X near X_obs)    = {post_p_H0:.4f}")
print(f"posterior odds H_0 : H_1           = {post_odds_H0_to_H1:.2f}")

MC trials                          = 5,000,000
filter window (X units)            = 512.10
trials with X near X_obs           = 460
  ... from H_0 (theta = 1/2)       = 436
  ... from H_1 (theta ~ U[0,1])    = 24
posterior P(H_0 | X near X_obs)    = 0.9478
posterior odds H_0 : H_1           = 18.17
